In [3]:
# ==============================================================================
# PHASE 2: INCREMENTAL AI CATEGORIZATION & CLEANUP
# 
# DESCRIPTION:
# This script identifies rows in the master dataset that lack a successful 
# categorization and processes them using Gemini 2.5 Flash. 
#
# REPRODUCIBILITY NOTE:
# - Uses structured JSON output to ensure schema consistency.
# - Implements a "Resume" feature to handle API timeouts or connection drops.
# - Pulls category definitions from 'config/categories.json'.
# ==============================================================================

import os
import sys
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from tqdm import tqdm
from dotenv import load_dotenv

# --- 1. SETUP & DIRECTORIES ---
# Establish paths relative to the notebook location
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
interim_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))

# Load environment variables (API Key)
load_dotenv(os.path.join(config_dir, ".env"))
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Define file paths
master_file = os.path.join(interim_dir, "master_ontology_dataset.csv")
mapping_file = os.path.join(interim_dir, "category_mapping_ai.csv")
category_config_file = os.path.join(config_dir, "categories.json")

# --- 2. LOAD CATEGORY CONFIGURATION ---
with open(category_config_file, "r") as f:
    CATEGORY_CONFIG = json.load(f)

ALLOWED_LABELS = [v['label'] for v in CATEGORY_CONFIG.values()]
MODEL_NAME = 'gemini-2.5-flash'

# --- 3. STRUCTURED OUTPUT SCHEMA ---
# This forces the AI to return data in a machine-readable format
response_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "curie": {"type": "string"},
            "category": {"type": "string", "enum": ALLOWED_LABELS},
            "confidence": {"type": "number"}
        },
        "required": ["curie", "category", "confidence"]
    }
}

# --- 4. SYSTEM INSTRUCTIONS ---
SYSTEM_INSTRUCTION = f"""
You are an expert ontologist specializing in the scientific study of religion. Your task is to categorize the 'Subject' concept.

DATA SCHEMA GUIDANCE:
- Subject (Primary_Label): This is the core concept to be categorized.
- supporting evidence: Use Synonyms, Hierarchy_Path, Source_System, and Description to clarify the meaning of the Subject.
- objective: Determine which category best fits the SUBJECT.

STRICT CATEGORY DEFINITIONS:
{json.dumps(CATEGORY_CONFIG, indent=2)}

STRICT DECISION RULES:
1. COMMUNITIES vs IDENTITIES:
   - Use 'communities' ONLY for GENERIC COMMON NOUNS representing types of social units (e.g., 'diocese', 'parish', 'congregation').
   - If the Subject is a PROPER NAME of a group (e.g., 'First Baptist Church', 'The Society of Jesus'), do NOT use 'communities'. Use 'identities' (if it represents a specific religious group) or 'religious other'.
"""

# --- 5. INITIALIZATION & RESUME LOGIC ---
# Load the master list of all concepts
master_df = pd.read_csv(master_file, dtype={'CURIE': str})

def get_todo_list():
    """Identifies rows that have not yet been successfully categorized."""
    if not os.path.exists(mapping_file):
        return master_df.copy()
    
    # Load existing progress
    current_mapping = pd.read_csv(mapping_file, dtype={'CURIE': str})
    # Filter for rows that succeeded
    processed_successfully = set(current_mapping[current_mapping['processing_status'] == 'Success']['CURIE'])
    
    # Return rows from master that are NOT in the success set
    return master_df[~master_df['CURIE'].isin(processed_successfully)].copy()

# --- 6. BATCH PROCESSING FUNCTION ---
def process_batch(chunk):
    """Sends a batch of concepts to the Gemini API."""
    prompt_items = []
    for _, r in chunk.iterrows():
        syns = str(r['Synonyms']) if pd.notnull(r['Synonyms']) else "None"
        desc = str(r['Description'])[:150] if pd.notnull(r['Description']) else "None"
        prompt_items.append(
            f"ID: {r['CURIE']} | Subject: {r['Primary_Label']} | Synonyms: {syns} | "
            f"Path: {r['Hierarchy_Path']} | Source: {r['Source_System']} | Desc: {desc}"
        )
    
    prompt_text = "Classify this batch of religious concepts:\n" + "\n".join(prompt_items)
    
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_text,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_INSTRUCTION,
                response_mime_type="application/json",
                response_schema=response_schema
            )
        )
        return response.parsed, "Success"
    except Exception as e:
        print(f"\n[!] API Batch Error: {e}")
        return None, "Failed"

# --- 7. MAIN EXECUTION LOOP ---
todo_df = get_todo_list()
BATCH_SIZE = 50  # Optimized for Flash model context window

if todo_df.empty:
    print("[*] All concepts are already categorized. No work to do.")
else:
    print(f"[*] Total Master Rows: {len(master_df)} | Remaining to Process: {len(todo_df)}")

    for i in tqdm(range(0, len(todo_df), BATCH_SIZE)):
        chunk = todo_df.iloc[i:i+BATCH_SIZE]
        results, status = process_batch(chunk)
        
        batch_results = []
        
        if status == "Success" and results:
            results_df = pd.DataFrame(results)
            results_df['curie'] = results_df['curie'].astype(str)
            
            # Re-align API results with the original chunk metadata
            merged = chunk.merge(results_df, left_on='CURIE', right_on='curie', how='left')
            
            for _, row in merged.iterrows():
                batch_results.append({
                    'CURIE': row['CURIE'],
                    'Primary_Label': row['Primary_Label'],
                    'category': row['category'],
                    'confidence': row['confidence'],
                    'Source_System': row['Source_System'],
                    'Hierarchy_Path': row['Hierarchy_Path'],
                    'processing_status': 'Success'
                })
        else:
            # If the batch fails, log it as 'Failed' for a future retry
            for _, row in chunk.iterrows():
                batch_results.append({
                    'CURIE': row['CURIE'],
                    'Primary_Label': row['Primary_Label'],
                    'category': None,
                    'confidence': 0,
                    'Source_System': row['Source_System'],
                    'Hierarchy_Path': row['Hierarchy_Path'],
                    'processing_status': 'Failed'
                })

        # Append this batch results to the persistent mapping file
        batch_output_df = pd.DataFrame(batch_results)
        file_exists = os.path.isfile(mapping_file)
        batch_output_df.to_csv(mapping_file, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
        
        # Rate limit protection
        time.sleep(1.2)

# --- 8. POST-PROCESSING CLEANUP ---
print("\n[*] Running integrity cleanup (deduplication)...")
if os.path.exists(mapping_file):
    final_map_df = pd.read_csv(mapping_file, dtype={'CURIE': str})
    
    # Sort: CURIE ascending, then Success over Failed, then highest Confidence
    final_map_df = final_map_df.sort_values(
        by=['CURIE', 'processing_status', 'confidence'], 
        ascending=[True, False, False]
    )
    
    # Keep only the single best categorization for each ID
    clean_map_df = final_map_df.drop_duplicates(subset=['CURIE'], keep='first')
    
    # Overwrite the mapping file with the clean version
    clean_map_df.to_csv(mapping_file, index=False, encoding='utf-8-sig')
    
    print(f"[COMPLETE] Mapping file clean and synchronized. Total rows: {len(clean_map_df)}")

[*] Total Master Rows: 8261 | Remaining to Process: 8261


  1%|          | 2/166 [01:26<1:57:51, 43.12s/it]


KeyboardInterrupt: 

In [4]:
# ==============================================================================
# STEP 2.1: INTEGRITY DIAGNOSTIC
# Evaluates the raw API output against the master dataset to ensure zero data 
# loss, zero hallucinations, and perfect text fidelity before human auditing.
# ==============================================================================
import os
import sys
import pandas as pd

print("\n" + "="*60)
print(" API OUTPUT INTEGRITY DIAGNOSTIC REPORT ")
print("="*60)

# Load data, forcing CURIE to string to prevent parsing differences
try:
    master_df = pd.read_csv(master_file, dtype={'CURIE': str})
    map_df = pd.read_csv(mapping_file, dtype={'CURIE': str})
except FileNotFoundError as e:
    print(f"[!] Error loading files: {e}")
    sys.exit(1)

print(f"Row Counts -> Master: {len(master_df)} | API Output: {len(map_df)}\n")

# --- A. CONCEPT COMPLETENESS ---
print("--- A. CONCEPT COMPLETENESS ---")
master_curies = set(master_df['CURIE'].dropna())
map_curies = set(map_df['CURIE'].dropna())

missing_from_api = master_curies - map_curies
hallucinated_by_api = map_curies - master_curies

print(f"Missing Concepts  : {len(missing_from_api)} (In master, absent from API output)")
if missing_from_api:
    print(f"  -> Examples: {list(missing_from_api)[:5]}")

print(f"Hallucinated IDs  : {len(hallucinated_by_api)} (Invented by API, absent from master)")
if hallucinated_by_api:
    print(f"  -> Examples: {list(hallucinated_by_api)[:5]}")

print("")

# --- B. DUPLICATE CONCEPTS ---
print("--- B. DUPLICATE CONCEPTS ---")
duplicates = map_df[map_df.duplicated(subset=['CURIE'], keep=False)]
duplicate_count = duplicates['CURIE'].nunique()

print(f"Distinct CURIEs duplicated: {duplicate_count}")

if duplicate_count > 0:
    print("  -> (Note: Duplicates are usually resolved during the Phase 2 post-processing cleanup.)")
    print("  -> Snapshot of unresolved duplicates:")
    for curie, group in duplicates.groupby('CURIE'):
        print(f"\nCURIE: {curie}")
        cols = [c for c in ['Primary_Label', 'category', 'processing_status'] if c in group.columns] 
        print(group[cols].to_string(index=False))

print("\n")

# --- C. TEXT FIDELITY (HALLUCINATION CHECK) ---
print("--- C. TEXT FIDELITY ---")
# Only check fidelity for CURIEs that exist in both datasets, dropping dupes for comparison
valid_map_df = map_df[~map_df['CURIE'].isin(hallucinated_by_api)].drop_duplicates(subset=['CURIE'], keep='first')

compare_df = master_df.merge(
    valid_map_df, 
    on='CURIE', 
    how='inner', 
    suffixes=('_master', '_api')
)

def check_mismatch(col_name):
    col_master = f"{col_name}_master"
    col_api = f"{col_name}_api"
    
    # Handle NaNs gracefully
    master_vals = compare_df[col_master].fillna('').astype(str).str.strip()
    api_vals = compare_df[col_api].fillna('').astype(str).str.strip()
    
    mismatches = compare_df[master_vals != api_vals]
    
    print(f"[{col_name}] Mismatches: {len(mismatches)}")
    if not mismatches.empty:
        print("  -> Examples of alteration:")
        for _, row in mismatches.head(3).iterrows():
            print(f"     Master: '{row[col_master]}' | API: '{row[col_api]}'")

if 'Primary_Label_api' in compare_df.columns:
    check_mismatch('Primary_Label')
if 'Source_System_api' in compare_df.columns:
    check_mismatch('Source_System')
if 'Hierarchy_Path_api' in compare_df.columns:
    check_mismatch('Hierarchy_Path')

print("\n" + "="*60)
print(" DIAGNOSTIC COMPLETE ")
print("="*60)


 API OUTPUT INTEGRITY DIAGNOSTIC REPORT 
Row Counts -> Master: 8261 | API Output: 100

--- A. CONCEPT COMPLETENESS ---
Missing Concepts  : 8161 (In master, absent from API output)
  -> Examples: ['AAT:300006562', 'AFSET:afset015179', 'AAT:300298862', 'ARDA:1265', 'AAT:300211673']
Hallucinated IDs  : 0 (Invented by API, absent from master)

--- B. DUPLICATE CONCEPTS ---
Distinct CURIEs duplicated: 0


--- C. TEXT FIDELITY ---
[Primary_Label] Mismatches: 0
[Source_System] Mismatches: 0
[Hierarchy_Path] Mismatches: 0

 DIAGNOSTIC COMPLETE 


In [ ]:
# ==============================================================================
# STEP 2.2: AI PERFORMANCE AUDIT REPORTING
# 
# DESCRIPTION:
# This script compares the original AI-assigned categories against the final 
# human-reviewed categories. It generates performance metrics to document the 
# reliability of the zero-shot LLM categorization for your methodology section.
#
# INPUT: 
# - data/interim/human_category_review.xlsx (The manually audited file)
#
# OUTPUTS (data/interim/):
# - report_changes_by_source.csv: Highlights which source ontologies the AI struggled with.
# - report_changes_by_category.csv: Highlights which specific categories were muddled.
# ==============================================================================

import os
import pandas as pd

print("\n" + "="*80)
print(" STEP 2.2: GENERATING AI PERFORMANCE REPORTS ")
print("="*80)

# --- 1. SETUP & DIRECTORIES ---
# Establish paths relative to the notebook location
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
interim_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))

# Define file paths based on directory structure
human_file = os.path.join(interim_data_dir, "human_category_review.xlsx")
report_source_file = os.path.join(interim_data_dir, "report_changes_by_source.csv")
report_category_file = os.path.join(interim_data_dir, "report_changes_by_category.csv")

# --- 2. LOAD & CLEAN DATA ---
print(f"[*] Loading audited dataset from: {os.path.basename(human_file)}...")
try:
    df = pd.read_excel(human_file)
except FileNotFoundError:
    print(f"[!] ERROR: Could not find {human_file}. Ensure your manual audit is saved and named correctly.")
    exit(1)

# Standardize text to prevent false positives (e.g., accidental trailing spaces from Excel)
# Using .fillna('') ensures that if a category was left blank, it doesn't break the string functions
df['category'] = df['category'].fillna('').astype(str).str.strip().str.lower()
df['category_human'] = df['category_human'].fillna('').astype(str).str.strip().str.lower()

# Create a boolean flag: True if the human changed the category, False if it remained the same
df['was_changed'] = df['category'] != df['category_human']

print(f"[*] Analyzing {len(df)} total concepts for manual overrides...")

# --- 3. TABLE 1: PERFORMANCE BY SOURCE SYSTEM ---
# This aggregation shows if the AI's logic works better for some ontologies (e.g., AAT) than others (e.g., DRH)
source_summary = df.groupby('Source_System').agg(
    Total_Concepts=('CURIE', 'count'),
    Unchanged=('was_changed', lambda x: (~x).sum()),
    Changed=('was_changed', 'sum')
).reset_index()

# Calculate the error/change rate per source
source_summary['%_Changed'] = (source_summary['Changed'] / source_summary['Total_Concepts'] * 100).round(1)
source_summary = source_summary.sort_values(by='%_Changed', ascending=False)

# --- 4. TABLE 2: PERFORMANCE BY ORIGINAL AI CATEGORY ---
# Grouping by the ORIGINAL AI category acts as a "confusion metric" to show which concepts 
# the model struggled to define (e.g., confusing identities with communities).
cat_summary = df.groupby('category').agg(
    Total_Assigned_By_AI=('CURIE', 'count'),
    Unchanged=('was_changed', lambda x: (~x).sum()),
    Changed=('was_changed', 'sum')
).reset_index()

# Calculate the error/change rate per category
cat_summary['%_Changed'] = (cat_summary['Changed'] / cat_summary['Total_Assigned_By_AI'] * 100).round(1)
cat_summary = cat_summary.sort_values(by='%_Changed', ascending=False)

# --- 5. DISPLAY AND EXPORT ---
print("\n" + "="*80)
print(" TABLE 1: AI CATEGORIZATION PERFORMANCE BY SOURCE SYSTEM")
print("="*80)
print(source_summary.to_string(index=False))

print("\n" + "="*80)
print(" TABLE 2: AI CATEGORIZATION PERFORMANCE BY ORIGINAL CATEGORY")
print("="*80)
print(cat_summary.to_string(index=False))

# Export to CSV for methodology documentation
source_summary.to_csv(report_source_file, index=False)
cat_summary.to_csv(report_category_file, index=False)

print("\n[*] Reports successfully generated and saved to 'data/interim/':")
print(f"    - {os.path.basename(report_source_file)}")
print(f"    - {os.path.basename(report_category_file)}")

[*] Loading human-reviewed dataset...

 TABLE 1: AI CATEGORIZATION PERFORMANCE BY SOURCE SYSTEM
                                         Source_System  Total_Concepts  Unchanged  Changed  %_Changed
       Logical Observation Identifiers Names and Codes             131         94       37       28.2
                        Office for National Statistics             241        217       24       10.0
                            AFS Ethnographic Thesaurus             997        897      100       10.0
                         Database of Religious History            1844       1666      178        9.7
                              Medical Subject Headings             114        106        8        7.0
                                             SNOMED CT             327        304       23        7.0
                  Library of Congress Subject Headings             510        478       32        6.3
                                                HL7 v3              82         78       